In [1]:
import os
import sys
import pandas as pd

# 1. RISOLUZIONE DEI PERCORSI
# Questo trucco dice a Python di guardare anche nella cartella radice (food_recommender/)
# così puoi importare config.py e la cartella models senza errori di "ModuleNotFoundError"
sys.path.append(os.path.abspath(os.path.join('..')))

from config import PATH_CLEAN_RECIPES, PATH_CLEAN_INTERACTIONS
from models.popularity import PopularityRecommender

print("Moduli importati con successo!")

# 2. CARICAMENTO DATI
# Usiamo os.path.join('..', ...) perché il notebook si trova dentro la cartella 'notebooks/'
# e deve fare un passo indietro per trovare il dataset
df_recipes = pd.read_csv(os.path.join('..', PATH_CLEAN_RECIPES))
df_interactions = pd.read_csv(os.path.join('..', PATH_CLEAN_INTERACTIONS))

print(f"Ricette caricate: {len(df_recipes)}")
print(f"Interazioni caricate: {len(df_interactions)}")

Moduli importati con successo!
Ricette caricate: 15144
Interazioni caricate: 385801


In [2]:
# 3. INIZIALIZZAZIONE E ADDESTRAMENTO
# Usiamo m=50 come richiesto dall'obiettivo (minimo 50 voti per considerare affidabile la media)
pop_model = PopularityRecommender(m=50)
pop_model.fit(df_recipes, df_interactions)

# 4. TEST 1: Le 5 ricette più popolari in assoluto
print("\n--- TOP 5 RICETTE POPOLARI IN ASSOLUTO ---")
top_global = pop_model.recommend(top_k=5)
for r in top_global:
    print(f"Nome: {r['name']} | Score Bayesiano: {r['score']} | Voti: {r['numero_voti']} | Rating Medio: {r['rating_medio']} | {r['calorie']} kcal")

# 5. TEST 2: Le 5 ricette più popolari filtrando per il tag 'low-carb'
print("\n--- TOP 5 RICETTE POPOLARI CON TAG 'LOW-CARB' ---")
top_low_carb = pop_model.recommend(tag='low-carb', top_k=5)
for r in top_low_carb:
    print(f"Nome: {r['name']} | Score Bayesiano: {r['score']} | Voti: {r['numero_voti']} | Rating Medio: {r['rating_medio']} | {r['calorie']} kcal")


--- TOP 5 RICETTE POPOLARI IN ASSOLUTO ---
Nome: pan release professional pan coating better than pam spray | Score Bayesiano: 4.9142 | Voti: 188 | Rating Medio: 4.96 | 1398.1 kcal
Nome: uncle bill s method for cooking turkey | Score Bayesiano: 4.9093 | Voti: 142 | Rating Medio: 4.97 | 675.3 kcal
Nome: beth s melt in your mouth barbecue ribs oven | Score Bayesiano: 4.9017 | Voti: 178 | Rating Medio: 4.95 | 1229.3 kcal
Nome: kittencal s best blasted rapid roast whole chicken | Score Bayesiano: 4.8999 | Voti: 224 | Rating Medio: 4.94 | 841.0 kcal
Nome: chicken pot pie with 2 crusts | Score Bayesiano: 4.8913 | Voti: 147 | Rating Medio: 4.95 | 562.4 kcal

--- TOP 5 RICETTE POPOLARI CON TAG 'LOW-CARB' ---
Nome: uncle bill s method for cooking turkey | Score Bayesiano: 4.9093 | Voti: 142 | Rating Medio: 4.97 | 675.3 kcal
Nome: kittencal s italian melt in your mouth meatballs | Score Bayesiano: 4.8852 | Voti: 746 | Rating Medio: 4.9 | 1312.6 kcal
Nome: kittencal s perfect prime rib roast bee

In [3]:
C_globale = df_interactions['rating'].mean()
print(f"Media globale del dataset (C): {C_globale:.4f}")

Media globale del dataset (C): 4.7317


In [4]:
ID_TEST = 78579  # <--- METTI QUI L'ID DELLA TUA PRIMA RICETTA IN OUTPUT

# Estraiamo i voti reali dati a questa ricetta
voti_ricetta = df_interactions[df_interactions['recipe_id'] == ID_TEST]['rating']

v_reale = len(voti_ricetta)
R_reale = voti_ricetta.mean()

print(f"Numero di voti reali (v): {v_reale}")
print(f"Rating medio reale (R): {R_reale:.4f}")

Numero di voti reali (v): 188
Rating medio reale (R): 4.9628


In [5]:
m = 50
score_manuale = (v_reale / (v_reale + m)) * R_reale + (m / (v_reale + m)) * C_globale
print(f"Score calcolato a mano: {score_manuale:.4f}")

Score calcolato a mano: 4.9142
